In [2]:
import pandas as pd
import glob
import calendar

# 월별 일자 데이터프레임 생성
days_lists = []
for month in range(1, 7):
    monthly = f'{month:02d}' + "월"
    daycnt = calendar.monthrange(2026, month)[1]
    days_lists.append({'월': monthly, '일자': daycnt})
daycnt = pd.DataFrame(days_lists)

# 일별 행정동 시간대 생활인구 병합처리
data_lists = glob.glob(r"D:\\02_사업관리\\자체분석\\01_생활인구\\hcode\\*.csv", recursive=True)
scode = pd.read_csv(r"D:\\02_사업관리\\코드자료\\부산구군코드.csv", dtype={'gcode': str}, encoding="utf-8")
# print(scode)
output_path = r"D:\\04_자료제출\\2026년\\2027년 자치구 일반조정교부금 산정 기초자료\\"
res_lists = []
for data in data_lists:
    df = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str, 'tme': str}, encoding="utf-8")
    df['연도'] = df['std_ymd'].astype('datetime64[ns]').astype(str).str[:4] + "년"
    df['월'] = df['std_ymd'].astype('datetime64[ns]').astype(str).str[5:7] + "월"
    df['gcode'] = df['hcode'].str[:5]
    df['sum_pop'] = df.loc[:, 'h_pop':'v_pop'].sum(axis=1)
    df['시간'] = df['tme'] + "시"
    df = pd.merge(df, scode, on='gcode', how="inner")
    df = df.sort_values(by='gcode', ascending=True)
    group = df.groupby(['월','gcode','gname','시간'], as_index=False)['sum_pop'].sum().sort_values(by=['월','gcode','시간'], ascending=True)
    group = pd.merge(group, daycnt, on="월", how='inner')
    group['mean_pop'] = round(group['sum_pop'] / group['일자'])
    group = group[['월', 'gcode', 'gname', '시간', 'mean_pop']]
    group.columns = ['월', '구군코드', '기관별', '시간', '인구수']
    res_lists.append(group)
result = pd.concat(res_lists, ignore_index=True)
pivoted = result.pivot_table(index=['구군코드','기관별'], columns=['월','시간'], values='인구수').reset_index().sort_values(by='구군코드', ascending=True)
pivoted.to_excel(output_path + "2026년_상반기_자치구별_생활인구_현황.xlsx", engine="openpyxl")